# DarkPipe 0.6 — AION seed-committed holdout replay

Reproducción completa del resultado `PASS_BOUNDED`: freeze, desafío sellado, predicción sin mapa y reveal verificable.

**Alcance:** siete frecuencias fijas, un holdout de controles AION reales y señales tangenciales sintéticas declaradas. No es una detección física ni una búsqueda continua.

In [ ]:
!git clone --depth 1 --branch v0.6.0 https://github.com/FacundoFirmenich/darkpipe-realdata.git
%cd darkpipe-realdata
!python -m pip install -q -e .

In [ ]:
!python -m pytest -q

In [ ]:
import json
from pathlib import Path
from darkpipe.aion import EVIDENCE_DIRECTORY
from darkpipe.aion_blind import (
    analyze_blind_challenge,
    prepare_blind_challenge,
    reveal_blind_challenge,
)

checked = Path("evidence/aion_blind_holdout_2026-08-25")
evidence = Path("evidence") / EVIDENCE_DIRECTORY
seed = json.loads((checked / "seed_reveal.json").read_text())["seed_hex"]
prereg = json.loads((checked / "sealed_manifest.json").read_text())["preregistration_commit"]
target = Path("/content/darkpipe_aion_blind_reproduction")

prepare_blind_challenge(evidence, target, seed, prereg)
blind = analyze_blind_challenge(evidence, target)
report = reveal_blind_challenge(target, seed)
report["decision"], report["gates"]

In [ ]:
checked_report = json.loads((checked / "report.json").read_text())
assert report["decision"] == checked_report["decision"] == "PASS_BOUNDED"
assert report["gates"] == checked_report["gates"]
assert [(x["label"], x["prediction"]["peak_dataset_id"], x["prediction"]["global_p"]) for x in report["cases"]] == [
    (x["label"], x["prediction"]["peak_dataset_id"], x["prediction"]["global_p"]) for x in checked_report["cases"]
]
print("Reproducción byte-governed y endpoints: OK")

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown((target / "report.md").read_text()))
display(Image(filename=str(target / "blind_validation.png")))

In [ ]:
import shutil
archive = shutil.make_archive("/content/DarkPipe_AION_Blind_v06_results", "zip", target)
print(archive)